In [11]:
import pandas as pd
from langcodes import Language
import uuid
from datasets import load_dataset, Dataset

In [3]:
seed_df = load_dataset("ljvmiranda921/msde-seed-S1", split="train").to_pandas()
print(seed_df.columns)

Generating train split: 100%|██████████| 449017/449017 [00:00<00:00, 691487.96 examples/s]


Index(['id', 'source', 'language', 'strategy', 'source_id', 'prompt',
       'response'],
      dtype='object')


In [4]:
LANG_MAPPING = {"Tagalog": "tl", "Filipino": "fil"}

Process Cohere Aya Collection

In [8]:
def _iso3_to_iso2(code: str) -> str:
    tag = Language.get(code).to_tag()
    iso2 = tag.split("-")[0]
    return iso2

In [12]:
aya_ds = load_dataset("CohereLabs/aya_collection", "aya_dataset", split="train")
filtered_df = aya_ds.to_pandas()
filtered_df["language"] = filtered_df["language"].apply(_iso3_to_iso2)
filtered_df = filtered_df[filtered_df["language"].isin(set(LANG_MAPPING.values()))].reset_index(drop=True)  # fmt: skip
aya_df = pd.DataFrame(
    {
        "id": [uuid.uuid4().hex for _ in range(len(filtered_df))],
        "source": "CohereLabs/aya_collection",
        "language": filtered_df["language"].astype(str).values,
        "prompt": filtered_df["inputs"].astype(str).values,
        "response": filtered_df["targets"].astype(str).values,
        "strategy": [["generate", "respond"] for _ in range(len(filtered_df))],
        "source_id": filtered_df["id"].astype(str).values,
    }
)

In [35]:
len(aya_df)

1241

Process AllenAI Wildchat

In [36]:
wildchat_df_raw = pd.read_json("data/tl.jsonl.zst", lines=True, compression="zstd")

In [42]:
wildchat_df = pd.DataFrame(
    {
        # fmt: off
        "id": [uuid.uuid4().hex for _ in range(len(wildchat_df_raw))],
        "source": wildchat_df_raw["source"].astype(str).values,
        "language": "tl",
        "prompt": [row[0]["value"] for _, row in wildchat_df_raw.conversations.items()],
        "response": [row[1]["value"] for _, row in wildchat_df_raw.conversations.items()],
        "strategy": [["generate", "respond"] for _ in range(len(wildchat_df_raw))],
        "source_id": "agentlans/allenai-WildChat_" + wildchat_df_raw.index.astype(str),
        # fmt: on
    }
)

Process Saillab Alpaca Training Dataset

In [44]:
alpaca_ds = load_dataset("saillab/alpaca-filipino-cleaned", split="train")

Generating test split: 100%|██████████| 10401/10401 [00:00<00:00, 1350952.43 examples/s]


In [ ]:
alpaca_df_raw = alpaca_ds.to_pandas()
alpaca_df_raw